### Mount drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/AML-Project/market.zip -d /content/Market-Pytorch

### Load libraries

In [ ]:
import torch
import torch.nn as nn
from torch.nn import init
import torch.optim as optim
from torchvision import models
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data.sampler import Sampler
from torchvision.models.vision_transformer import vit_b_16
from torchvision.models import ViT_B_16_Weights

import os
import shutil
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import timm
import json
import numpy as np
from PIL import Image
import copy
import random
import itertools
from collections import defaultdict
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(12)
np.random.seed(12)
random.seed(12)

# Contrastive Transformers (Training)

### Training

In [ ]:
class PairDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.classes = list(data.keys())
        self.transform = transform

    def __len__(self):
        return len(self.classes)

    def __getitem__(self, index):
        class_label = self.classes[index]
        images = self.data[class_label]
        img1, img2 = random.sample(images, 2)

        if self.transform:
            img1 = self.transform(Image.open(img1))
            img2 = self.transform(Image.open(img2))

        return img1, img2

class AllPairDataset(Dataset):
  def __init__(self, files, transform = None):
    self.files = files
    self.transform = transform

  def __len__(self):
    return len(self.files)

  def __getitem__(self, idx):
    image1 = Image.open(self.files[idx][0]).convert('RGB')
    image2 = Image.open(self.files[idx][1]).convert('RGB')

    if self.transform:
      image1 = self.transform(image1)
      image2 = self.transform(image2)

    return image1, image2

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, embed_dim = 768):
        super(ProjectionHead, self).__init__()

        self.multihead_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)

        # Layer Normalization
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

        # Feedforward with GELU Activation
        self.feedforward = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.GELU(),
            nn.Linear(1024, embed_dim)
        )

        # Adaptive Average Pooling
        self.gem = GeMPooling(output_size = 5)

        self.dropout_attn = nn.Dropout(p=0.1)
        self.dropout_ff = nn.Dropout(p=0.1)

    def forward(self, x):
        attn_output, _ = self.multihead_attn(x, x, x)  # [BATCH, 197, 768]
        attn_output = self.dropout_attn(attn_output)
        x = x + attn_output  # Residual Connection
        x = self.layer_norm1(x)  # Layer Norm

        # Feedforward Layer
        ff_output = self.feedforward(x)  # [BATCH, 197, 768]
        ff_output = self.dropout_ff(ff_output)
        x = x + ff_output  # Residual Connection
        x = self.layer_norm2(x)  # Layer Norm

        x = self.gem(x)  # [BATCH, 197, 5]

        # Permute back and average: [BATCH, 985]
        x = x.view(x.size(0), -1)

        return x

class ContrastiveTransformer(nn.Module):
    def __init__(self, freeze=True):
        super(ContrastiveTransformer, self).__init__()
        self.backbone = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        if freeze:
          for param in self.backbone.parameters():
            param.requires_grad = False
        self.projection_head = ProjectionHead()

    def forward(self,x):
        x = self.backbone._process_input(x)
        batch_class_token = self.backbone.class_token.expand(x.shape[0], -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.backbone.encoder(x)

        x = self.projection_head(x)
        return x

class GeMPooling(nn.Module):
    def __init__(self, p=3.0, output_size=1):
        super(GeMPooling, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.output_size = output_size

    def forward(self, x):
        return F.adaptive_avg_pool1d(x.clamp(min=1e-6).pow(self.p), self.output_size).pow(1.0 / self.p)

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.model1 = ContrastiveTransformer()
        self.model2 = self.model1

    def forward(self, x1, x2 = None):
        y1 = self.model1(x1)

        if x2 is None:
          return y1
        else:
          y2 = self.model2(x2)

        return y1, y2

In [ ]:
class UniqueClassBatchSampler(Sampler):
    def __init__(self, data, batch_size):
        self.classes = list(data.keys())
        self.batch_size = batch_size

    def __iter__(self):
        class_indices = list(range(len(self.classes)))
        random.shuffle(class_indices)

        for i in range(0, len(class_indices), self.batch_size):
            if (i + self.batch_size) > len(class_indices):
                yield class_indices[i:]
            else:
                batch_indices = class_indices[i:i+self.batch_size]
                yield batch_indices

    def __len__(self):
        return math.ceil(len(self.classes) // self.batch_size)


def get_data(data_dir="/content/Market-Pytorch/Market/train", all_pairs = False):

  files = defaultdict(list)

  for folder_name in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder_name)
    images_path = [os.path.join(folder_path, image_name) for image_name in os.listdir(folder_path)]
    if len(images_path) > 1:
      files[folder_name] = images_path

  val_split, train_split = 0.2, 0.8
  class_keys = list(files.keys())
  random.shuffle(class_keys)
  split_index = int(len(class_keys) * (1 - val_split))
  train_keys = class_keys[:split_index]
  val_keys = class_keys[split_index:]

  train_data = {key: files[key] for key in train_keys}
  val_data = {key: files[key] for key in val_keys}

  batch_size = 32

  if all_pairs:
    train_files = []
    val_files = []
    for k, v in train_data.items():
      couples = list(itertools.combinations(v, 2))
      train_files.extend(couples)

    for k,v in val_data.items():
      couples = list(itertools.combinations(v, 2))
      val_files.extend(couples)

    transform_train_list = transforms.Compose([
      transforms.Resize((224,224), interpolation=3),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    transform_val_list = transforms.Compose([
      transforms.Resize(size=(224,224),interpolation=3), #Image.BICUBIC
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    random.shuffle(train_files)
    random.shuffle(val_files)

    dataset_train = AllPairDataset(train_files, transform = transform_train_list)
    dataset_val = AllPairDataset(val_files, transform = transform_val_list)
    train_loader = DataLoader(dataset = dataset_train, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset = dataset_val, batch_size=batch_size, shuffle=True)

  else:

    preprocessing = ViT_B_16_Weights.DEFAULT.transforms()

    dataset_train = PairDataset(train_data, transform = preprocessing)
    dataset_val = PairDataset(val_data, transform = preprocessing)

    train_loader = DataLoader(dataset = dataset_train, batch_sampler = UniqueClassBatchSampler(train_data, batch_size))
    val_loader = DataLoader(dataset = dataset_val, batch_sampler = UniqueClassBatchSampler(val_data, batch_size))

  return train_loader, val_loader

In [ ]:
def get_training_objects(pretraining_path = None, freeze = True):
  model = SiameseNetwork(freeze = freeze).to(device)

  if pretraining_path:
    model.load_state_dict(torch.load(pretraining_path), strict=False)

  optimizer = optim.AdamW(model.parameters(), weight_decay=5e-4, lr=3e-4)

  return model, optimizer

In [ ]:
def contrastive_loss(features_1, features_2, temperature=0.07):

    features_1 = F.normalize(features_1, p=2, dim=1)
    features_2 = F.normalize(features_2, p=2, dim=1)

    logits = torch.matmul(features_1, features_2.t()) / temperature
    labels = torch.arange(features_1.size(0)).to(device)
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2
    return loss, logits

def train(num_epochs, model, train_loader, val_loader, optimizer, save_model_path, log_file):

    os.makedirs(save_model_path, exist_ok=True)
    log_data = []

    with tqdm(total=num_epochs, desc='Total Progress', unit='epoch') as epoch_pbar:
      for epoch in range(num_epochs):

          model.train()
          train_loss_total = 0.0
          total_correct = 0.0
          total_predictions = 0.0
          train_samples_total = 0

          with tqdm(train_loader, unit="batch") as pbar:
              for data1, data2 in pbar:
                  data1, data2 = data1.to(device), data2.to(device)

                  optimizer.zero_grad()
                  output1, output2 = model(data1, data2)

                  loss, logits = contrastive_loss(output1, output2)
                  loss.backward()
                  optimizer.step()

                  batch_size = logits.size(0)

                  train_loss_total += loss.item() * batch_size
                  train_samples_total += batch_size

                  labels = torch.arange(batch_size).to(device)

                  preds1 = logits.argmax(dim=1)
                  correct1 = (preds1 == labels).sum().item()

                  preds2 = logits.argmax(dim=0)
                  correct2 = (preds2 == labels).sum().item()

                  total_correct += (correct1 + correct2)
                  total_predictions += (batch_size * 2)

                  current_loss = train_loss_total / train_samples_total
                  current_accuracy = total_correct / total_predictions

                  pbar.set_description(f"Epoch {epoch+1}/{num_epochs}")
                  pbar.set_postfix(loss=current_loss, accuracy=current_accuracy)

          avg_train_loss = train_loss_total / train_samples_total
          avg_train_accuracy = total_correct / total_predictions

          model.eval()
          val_loss_total = 0.0
          val_correct = 0
          val_predictions = 0
          val_samples_total = 0

          with torch.no_grad():
              with tqdm(val_loader, unit="batch") as val_pbar:
                  for data1, data2 in val_pbar:
                      data1, data2 = data1.to(device), data2.to(device)

                      output1, output2 = model(data1, data2)

                      loss, logits = contrastive_loss(output1, output2)

                      batch_size = logits.size(0)

                      val_loss_total += loss.item() * batch_size
                      val_samples_total += batch_size

                      labels = torch.arange(batch_size).to(device)

                      # Compute number of correct predictions for acc1
                      preds1 = logits.argmax(dim=1)
                      correct1 = (preds1 == labels).sum().item()

                      # Compute number of correct predictions for acc2
                      preds2 = logits.argmax(dim=0)
                      correct2 = (preds2 == labels).sum().item()

                      # Update validation correct and predictions
                      val_correct += (correct1 + correct2)
                      val_predictions += (batch_size * 2)

                      # Compute current validation loss and accuracy
                      current_val_loss = val_loss_total / val_samples_total
                      current_val_accuracy = val_correct / val_predictions

                      val_pbar.set_description(f"Epoch {epoch+1}/{num_epochs} [Val]")
                      val_pbar.set_postfix(loss=current_val_loss, accuracy=current_val_accuracy)

          # Compute average loss and accuracy for the validation epoch
          avg_val_loss = val_loss_total / val_samples_total
          avg_val_accuracy = val_correct / val_predictions

          model_save_path = os.path.join(save_model_path, f'model_epoch_{epoch+1}.pth')
          torch.save(model.state_dict(), model_save_path)

          epoch_log = {
              'epoch': epoch + 1,
              'training_loss': avg_train_loss,
              'training_accuracy': avg_train_accuracy,
              'val_loss': avg_val_loss,
              'val_accuracy': avg_val_accuracy
          }
          log_data.append(epoch_log)

          # Save log data to JSON file
          with open(log_file, 'w') as f:
              json.dump(log_data, f, indent=4)

          print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f} - Validation Loss: {avg_val_loss:.4f} - Training Accuracy: {avg_train_accuracy:.4f} - Validation Accuracy: {avg_val_accuracy:.4f}")

          epoch_pbar.update(1)
          epoch_pbar.set_postfix(
                train_loss=avg_train_loss,
                train_accuracy=avg_train_accuracy,
                val_loss=avg_val_loss,
                val_accuracy=avg_val_accuracy
          )
          print("==================================================================================")

In [ ]:
def train_model(save_path = "/content/drive/MyDrive/AML-Project/Constrastive_Transformers/", all_pairs = False, freeze=True):
  train_loader, val_loader = get_data(all_pairs = all_pairs)
  model, optimizer = get_training_objects(freeze = freeze)
  train(1000, model, train_loader, val_loader, optimizer, save_path, os.path.join(save_path, "training_results.json"))

In [ ]:
train_model(save_path = "/content/drive/MyDrive/AML-Project/Constrastive_Transformers", freeze=False)